# 🌐 Translation Experiment - Piecewise Linear Attention

Compare **Standard**, **Linear**, and **Piecewise** attention in an encoder-decoder transformer on German→English translation.

## What This Does

- Trains encoder-decoder transformer on Multi30K dataset
- Compares all three attention mechanisms
- Measures training time and translation quality
- Generates comparison plots and tables

## Expected Runtime

- **With GPU (T4)**: ~15-30 minutes (5 epochs)
- **Without GPU**: ~2-4 hours (not recommended)

---

## 📦 Setup

In [ ]:
# Configuration
REPO_URL = "https://github.com/grapentt/piecewise-linear-attention.git"
BRANCH = "main"  # Change to your branch if needed

# Clone repository
import os
if not os.path.exists("piecewise-linear-attention"):
    print(f"Cloning repository (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already exists")

%cd piecewise-linear-attention
!git pull origin {BRANCH}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -e ".[experiments]" -q
print("✓ Installation complete!")

In [ ]:
# Check GPU
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("⚠ No GPU - using CPU (will be much slower)")
    print("  Enable GPU: Runtime → Change runtime type → GPU")

print(f"\nDevice: {device}")

---

## 🎯 Experiment Configuration

In [ ]:
# Experiment settings
ATTENTION_TYPES = ["standard", "linear", "piecewise"]  # Which attention types to compare
EPOCHS = 5  # Number of training epochs
BATCH_SIZE = 32  # Batch size
NUM_LAYERS = 2  # Number of encoder/decoder layers
HIDDEN_DIM = 256  # Hidden dimension
NUM_HEADS = 4  # Number of attention heads
MAX_TRAIN_SAMPLES = 5000  # Training samples (use 29000 for full dataset)
MAX_EVAL_SAMPLES = 500  # Eval samples (use 1000 for full dataset)

print("Experiment Configuration:")
print(f"  Attention types: {', '.join(ATTENTION_TYPES)}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Model: {NUM_LAYERS} layers, {HIDDEN_DIM}d, {NUM_HEADS} heads")
print(f"  Dataset: {MAX_TRAIN_SAMPLES} train, {MAX_EVAL_SAMPLES} eval")
print(f"  Device: {device}")

---

## 🚀 Run Experiment

In [ ]:
# Run translation experiment
!python experiments/translation/train.py \
  --attention-types {' '.join(ATTENTION_TYPES)} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --num-layers {NUM_LAYERS} \
  --hidden-dim {HIDDEN_DIM} \
  --num-heads {NUM_HEADS} \
  --max-train-samples {MAX_TRAIN_SAMPLES} \
  --max-eval-samples {MAX_EVAL_SAMPLES} \
  --device {device} \
  --save results/translation/colab_results.json

print("\n" + "="*80)
print("✓ Translation experiment complete!")
print("="*80)

---

## 📊 Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open("results/translation/colab_results.json") as f:
    results = json.load(f)

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Perplexity over epochs
ax = axes[0]
for result in results:
    attn = result["attention_type"]
    history = result["eval_history"]
    perplexities = [epoch["perplexity"] for epoch in history]
    ax.plot(perplexities, label=attn.capitalize(), linewidth=2, marker='o')

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Perplexity (lower is better)", fontsize=12)
ax.set_title("Translation Quality", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Training time
ax = axes[1]
attention_types = [r["attention_type"].capitalize() for r in results]
train_times = [r["total_train_time"] for r in results]
colors = ["#e74c3c", "#3498db", "#2ecc71"]
bars = ax.bar(attention_types, train_times, color=colors[:len(results)])

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel("Training Time (seconds)", fontsize=12)
ax.set_title("Training Time", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Speedup
ax = axes[2]
baseline_time = next(r["total_train_time"] for r in results if r["attention_type"] == "standard")
speedups = [baseline_time / r["total_train_time"] for r in results]
bars = ax.bar(attention_types, speedups, color=colors[:len(results)])

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Baseline')
ax.set_ylabel("Speedup vs Standard", fontsize=12)
ax.set_title("Speedup", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle("Translation Experiment (German→English)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Print summary table
print("\n" + "="*100)
print("RESULTS SUMMARY")
print("="*100)
print(f"{'Attention':<15} {'Perplexity':<15} {'Loss':<12} {'Train Time (s)':<18} {'Speedup':<10}")
print("-"*100)

baseline_time = next(r["total_train_time"] for r in results if r["attention_type"] == "standard")

for result in results:
    attn = result["attention_type"].capitalize()
    ppl = result["final_eval_metrics"]["perplexity"]
    loss = result["final_eval_metrics"]["loss"]
    train_time = result["total_train_time"]
    speedup = baseline_time / train_time
    print(f"{attn:<15} {ppl:<15.2f} {loss:<12.4f} {train_time:<18.1f} {speedup:<10.2f}×")

print("="*100)

# Key insights
piecewise = next(r for r in results if r["attention_type"] == "piecewise")
standard = next(r for r in results if r["attention_type"] == "standard")
speedup = baseline_time / piecewise["total_train_time"]
ppl_diff = ((piecewise["final_eval_metrics"]["perplexity"] / standard["final_eval_metrics"]["perplexity"]) - 1) * 100

print(f"\n✨ Key Findings:")
print(f"  • Piecewise is {speedup:.2f}× faster than Standard")
print(f"  • Perplexity: {ppl_diff:+.1f}% vs Standard ({'+' if ppl_diff > 0 else ''}{ppl_diff:.1f}% is worse)")
print(f"  • Trade-off: ~{speedup:.1f}× speedup with ~{abs(ppl_diff):.1f}% quality change")

---

## 💾 Download Results

In [ ]:
from google.colab import files

print("Downloading results...")
files.download("results/translation/colab_results.json")
print("✓ Download complete!")

---

## 📚 Learn More

- [Translation Experiment Guide](https://github.com/grapentt/piecewise-linear-attention/blob/main/experiments/translation/README.md)
- [Main README](https://github.com/grapentt/piecewise-linear-attention/blob/main/README.md)
- [Theory](https://github.com/grapentt/piecewise-linear-attention/blob/main/THEORY.md)

**Happy Experimenting! 🚀**